In [ ]:
# 1 工具调用的方式
## 方式1：直接调用

In [ ]:
from langchain_core.tools import tool
@tool
def get_weather(city: str) -> str:
    """
    获取指定城市的天气信息
    参数:
    city: 城市名称，如"北京"、"上海"
    返回:
    天气信息字符串
    """
    # 你的实现
    return city + "晴天，温度 15°C"

In [ ]:
# 使用 .invoke() 方法
result = get_weather.invoke({"city": "北京"})
print(result)

## 方式2：绑定到模型（主流）

In [1]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv(override=True)
DEEPSEEK_API_KEY= os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL= os.getenv("DEEPSEEK_BASE_URL")
llmOpenai = init_chat_model(
    # model="deepseek-v4-flash",
    # model_provider="deepseek",
    # 或者
    model="deepseek:deepseek-v4-flash",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
)


content='你好！很高兴认识你！😊\n\n我是 **DeepSeek**，由深度求索公司创造的AI助手。让我简单介绍一下自己：\n\n## 🎯 我的核心能力\n\n- **智能对话**：我可以回答问题、进行讨论、提供建议\n- **知识广博**：知识截止到2025年5月，涵盖各个领域\n- **多语言支持**：能用你使用的语言流畅交流\n- **文件处理**：支持上传图片、PDF、Word、Excel、PPT等文件，并读取其中的文字信息\n- **长文本处理**：拥有1M的上下文窗口，可以一次性处理像《三体》三部曲那么大的内容！\n\n## ✨ 我的特点\n\n- **完全免费**：没有任何收费计划，放心使用\n- **联网搜索**：需要时你可以在Web/App端手动开启联网功能\n- **语音输入**：App端支持语音输入，更方便交流\n- **纯文本模型**：虽然不能直接“看”图片，但可以读取图片中的文字\n\n## 📱 如何使用我\n\n你可以通过官方应用商店下载App，或者直接在网页上使用我。有任何问题、需要帮助，或者只是想聊聊天，我都在这里！\n\n**有什么我可以帮你的吗？** 无论是学习、工作还是日常问题，尽管问我！💪' additional_kwargs={'refusal': None, 'reasoning_content': '好的，用户让我介绍自己。这是一个非常常见且简单的开场问题。用户可能是初次接触，想了解我的基本身份和能力，以便决定如何与我互动。\n\n我需要给出一个清晰、友好、全面的自我介绍，涵盖我的身份、核心功能、特点以及使用方式。想到了先用问候语建立亲切感，然后直接说明我是DeepSeek，由深度求索公司创造。接着列出主要能力，比如文本处理、文件支持、联网搜索等，并强调免费和长上下文的特点。最后用总结和开放式的邀请结束，让用户知道可以继续提问。\n\n嗯，回复结构可以这样：问候与身份说明 -> 分点介绍关键特性（能力、支持格式、上下文长度、费用、平台） -> 总结并邀请提问。这样既全面又不会太冗长，符合用户对初次介绍快速获取信息的预期。'} response_metadata={'token_usage': {'completion_tokens': 453, 'prompt_tokens': 6, 'total_tokens

In [6]:
from langchain_core.tools import tool

# 定义工具
@tool
def get_weather(city: str) -> str:
    """获取指定城市的天气"""
    # 你的实现
    return "晴天，温度 15°C"

# 绑定工具
model_with_tools = llmOpenai.bind_tools([get_weather])

response = model_with_tools.invoke("上海今天天气怎么样")

if response.tool_calls:
    print("需要调用tool",response.tool_calls)
else:
    print("不需要调用tool",response.content)

需要调用tool [{'name': 'get_weather', 'args': {'city': '上海'}, 'id': 'call_00_UeOYXnN4CC733ugbLVNU5001', 'type': 'tool_call'}]


## 1.4 从Message流转看工具的调用

In [7]:
from langchain.messages import HumanMessage, ToolMessage
@tool
def get_weather(city: str):
    """获取天气的工具"""
    return f"{city}天气晴朗~"
# 将模型和工具绑定
model_with_tools = llmOpenai.bind_tools([get_weather])
messages = [
    HumanMessage("今天北京天气如何")
]
# 模型生成调用工具请求
response = model_with_tools.invoke(messages)
# 添加AIMessage
messages.append(response)

tool_calls = response.tool_calls
for tool_call in tool_calls:
    if tool_call["name"] == "get_weather":
        # 返回的是ToolMessage类型消息
        tool_response = get_weather.invoke(tool_call)
        print(type(tool_response))
        messages.append(tool_response)
print("=====================> messages <=====================")
for msg in messages:
    msg.pretty_print()
print("=====================> messages <=====================")
final_response = model_with_tools.invoke(messages)
print(f"final_response: \n{final_response}")

<class 'langchain_core.messages.tool.ToolMessage'>
=====================> messages <=====================
================================ Human Message =================================

今天北京天气如何
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_00_sjhI5rVQqADEs96JyBGK0485)
 Call ID: call_00_sjhI5rVQqADEs96JyBGK0485
  Args:
    city: 北京
================================= Tool Message =================================
Name: get_weather

北京天气晴朗~
=====================> messages <=====================
final_response: 
content='今天北京的天气是 **晴朗** ☀️，是个好天气！如果有出行计划的话，很适合户外活动哦～' additional_kwargs={'refusal': None, 'reasoning_content': '工具返回的结果是"北京天气晴朗~"，我来告诉用户。'} response_metadata={'token_usage': {'completion_tokens': 40, 'prompt_tokens': 353, 'total_tokens': 393, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 14, 'rejected_prediction_tokens': None}, 'prompt_tokens_details':